
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>




# Orchestrating Jobs with Databricks Workflows

New updates to the Databricks Jobs UI have added the ability to schedule multiple tasks as part of a job, allowing Databricks Jobs to fully handle orchestration for most production workloads.

Here, we'll start by reviewing the steps for scheduling a notebook task as a triggered standalone job, and then add a dependent task using a DLT pipeline. 

## Learning Objectives
By the end of this lesson, you should be able to:
* Schedule a notebook task in a Databricks Workflow Job
* Describe job scheduling options and differences between cluster types
* Review Job Runs to track progress and see results
* Schedule a DLT pipeline task in a Databricks Workflow Job
* Configure linear dependencies between tasks using the Databricks Workflows UI

In [0]:
%run ../Includes/Classroom-Setup-05.1.1


## Generate Job Configuration

Configuring this job will require parameters unique to a given user.

Run the cell below to print out values you'll use to configure your pipeline in subsequent steps.

In [0]:
DA.print_job_config_v1()


## Configure Job with a Single Notebook Task

When using the Jobs UI to orchestrate a workload with multiple tasks, you'll always begin by creating a job with a single task.

Steps:
1. Click the **Workflows** button on the sidebar, click the **Jobs** tab, and click the **Create Job** button.
2. Configure the job and task as specified below. You'll need the values provided in the cell output above for this step.

| Setting | Instructions |
|--|--|
| Task name | Enter **Reset** |
| Type | Choose **Notebook** |
| Source | Choose **Workspace** |
| Path | Use the navigator to specify the **Reset Notebook Path** provided above |
| Cluster | From the dropdown menu, under **Existing All Purpose Clusters**, select your cluster |
| Job name | In the top-left of the screen, enter the **Job Name** provided above to add a name for the job (not the task) |

<br>

3. Click the **Create** button.
4. Click the blue **Run now** button in the top right to start the job.

<img src="https://files.training.databricks.com/images/icon_note_24.png"> **Note**: When selecting your all-purpose cluster, you will get a warning about how this will be billed as all-purpose compute. Production jobs should always be scheduled against new job clusters appropriately sized for the workload, as this is billed at a much lower rate.

In [0]:
# This function is provided for students who do not 
# want to work through the exercise of creating the job.
DA.create_job_v1()

In [0]:
DA.validate_job_v1_config()



## Explore Scheduling Options
Steps:
1. On the right hand side of the Jobs UI, locate the **Job Details** section.
1. Under the **Trigger** section, select the **Add trigger** button to explore scheduling options.
1. Changing the **Trigger type** from **None (Manual)** to **Scheduled** will bring up a cron scheduling UI.
   - This UI provides extensive options for setting up chronological scheduling of your Jobs. Settings configured with the UI can also be output in cron syntax, which can be edited if custom configuration not available with the UI is needed.
1. At this time, we'll leave our job set to **Manual** scheduling; select **Cancel** to return to Job details.



## Review Run

To review the job run:
1. On the Jobs details page, select the **Runs** tab in the top-left of the screen (you should currently be on the **Tasks** tab)
1. Find your job.
    - If **the job is still running**, it will be under the **Active runs** section. 
    - If **the job finished running**, it will be under the **Completed runs** section
1. Open the output details by clicking on the timestamp field under the **Start time** column
    - If **the job is still running**, you will see the active state of the notebook with a **Status** of **`Pending`** or **`Running`** in the right side panel. 
    - If **the job has completed**, you will see the full execution of the notebook with a **Status** of **`Succeeded`** or **`Failed`** in the right side panel
  
The notebook employs the magic command **`%run`** to call an additional notebook using a relative path. Note that while not covered in this course, <a href="https://docs.databricks.com/repos.html#work-with-non-notebook-files-in-a-databricks-repo" target="_blank">new functionality added to Databricks Repos allows loading Python modules using relative paths</a>.

The actual outcome of the scheduled notebook is to reset the environment for our new job and pipeline.

## Generate Pipeline

In this step, we'll add a DLT pipeline to execute after the success of the task we configured at the start of this lesson.

To focus on jobs and not pipelines, we are going to use the following utility command to create a simple pipeline for us.

In [0]:
DA.create_pipeline()


## Configure a DLT Pipeline Task

Next, we need to add the task to run this pipeline.

Steps:
1. On the Job details page, click the **Tasks** tab.
1. Click the blue **+ Add task** button at the center bottom of the screen, and select **Delta Live Tables pipeline** in the dropdown menu.
1. Configure the task as specified below.

| Setting | Instructions |
|--|--|
| Task name | Enter **DLT** |
| Type | Leave **Delta Live Tables pipeline** |
| Pipeline | Choose the DLT pipeline configured above |
| Depends on | Choose **Reset**, which is the previous task we defined |

<br>

4. Click the blue **Create task** button
    - You should now see a screen with 2 boxes and a downward arrow between them. 
    - Your **`Reset`** task will be at the top, leading into your **`DLT`** task. 
    - This visualization represents the dependencies between these tasks.
5. Validate the configuration by running the command below.
    - If errors are reported, repeat the following until all errors have been removed.
      - Fix the error(s).
      - Click the **Create task** button.
      - Validate the configuration.

In [0]:
# This function is provided for students who do not 
# want to work through the exercise of creating the job.
DA.create_job_v2()

In [0]:
DA.validate_job_v2_config()


## Run the job
Once the job has been properly configured, click the blue **Run now** button in the top right to start the job.
<img src="https://files.training.databricks.com/images/icon_note_24.png"> **Note**: When selecting your all-purpose cluster, you will get a warning about how this will be billed as all-purpose compute. Production jobs should always be scheduled against new job clusters appropriately sized for the workload, as this is billed at a much lower rate.

**NOTE**: You may need to wait a few minutes as infrastructure for your job and pipeline is deployed.

In [0]:
# This function is provided to start the pipeline and  
# block until it has completed, canceled or failed
DA.start_job()



## Review Multi-Task Run Results

To review run results:
1. On the Job details page, select the **Runs** tab again and then the most recent run under **Active runs** or **Completed runs** depending on if the job has completed or not.
    - The visualizations for tasks will update in real time to reflect which tasks are actively running, and will change colors if task failures occur. 
1. Clicking on a task box will render the scheduled notebook in the UI. 
    - You can think of this as just an additional layer of orchestration on top of the previous Databricks Jobs UI, if that helps;
    - Note that if you have workloads scheduling jobs with the CLI or REST API, <a href="https://docs.databricks.com/dev-tools/api/latest/jobs.html" target="_blank">the JSON structure used to configure and get results about jobs has seen similar updates to the UI</a>.

**NOTE**: At this time, DLT pipelines scheduled as tasks do not directly render results in the Runs GUI; instead, you will be directed back to the DLT Pipeline GUI for the scheduled Pipeline.


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>